# IMV during GNN training: six graph benchmarks

This is a documented reconstruction of Section 6 / Figure 5 and Appendix F.3 of *Mastering Model Fit: Applications of the InterModel Vigorish for Interpretable Machine Learning*. In the supplied PDF, the appendix learning curves are **Figure A6**, not A5.

The six examples are **PROTEINS, NCI1, NCI109, Mutagenicity, AIDS, and DD**. All are the original public [TU graph datasets](https://chrsmrrs.github.io/datasets/docs/datasets/), not tabular substitutes or synthetic stand-ins. This notebook produces results only. Figures are rebuilt separately in `src/plotter/plotter.ipynb`, together with the ablation overview.


## Reconstruction protocol

| Item | Protocol | Evidence / choice |
|---|---|---|
| Architecture | Three GCN layers | Paper Section 5.1.3 |
| Training | Adam, learning rate 0.001, logical batch size 512, 200 epochs | Paper Section 5.1.3 and Figure 5; Adam is an explicit implementation choice |
| Split | Stratified 70% train / 15% validation / 15% test, split anew for each seed | Appendix D.4; stratification and extension to appendix datasets are explicit choices |
| Replicates | Seeds 42-51, ten independent training runs per dataset | Requested extension from the paper's five runs |
| Unspecified architecture | Hidden width 64, ReLU after each GCN, global mean pooling, dropout 0.5 before a one-logit classifier, no weight decay | Fixed in advance; not tuned on test results |
| Features | Provided one-hot node labels plus continuous node attributes; continuous columns standardized using training nodes only | [PyG TUDataset](https://pytorch-geometric.readthedocs.io/en/2.7.0/generated/torch_geometric.datasets.TUDataset.html); no outcome features |
| Optional engineering | `native+degree` appends standardized log(1 + degree) | Separate, fingerprinted experiment; not the default reconstruction |
| Graph structure | Undirected unweighted edges, duplicate removal, one self-loop per node, symmetric GCN normalization | [GCNConv definition](https://pytorch-geometric.readthedocs.io/en/2.7.0/generated/torch_geometric.nn.conv.GCNConv.html) |
| Edge labels | Not used by this ordinary GCN | Explicit limitation for molecular datasets with bond labels |
| Epoch evaluation | Validation and test predictions at epoch 0 and every epoch 1-200 | Epoch 0 is an added untrained-model diagnostic |
| Selection | Fixed 200 epochs; no early stopping, test-based tuning, or best-test-epoch selection | Prevents test curves from feeding back into model fitting |

Original TU datasets can contain isomorphic/duplicate graphs. We retain the original rather than the cleaned variants for comparability with the paper; random graph splits are not a scaffold split or a guarantee against isomorphism bias. Raw-file hashes, exact graph split IDs, feature scaling, label mapping, library versions, and configuration are recorded for every seed. Exact numerical reproduction is not claimed because the paper does not specify all implementation details.


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "empirical" / "gnn" / "gnn_training.py").is_file()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import torch
import imvpy
from empirical.gnn.gnn_training import DATASETS, SEEDS, GNNConfig, cache_paths, execution_plan, run_experiments

CONFIG = GNNConfig()
DEVICE = os.environ.get("IMV_GNN_DEVICE", "auto")
JOBS = int(os.environ["IMV_GNN_JOBS"]) if "IMV_GNN_JOBS" in os.environ else None
FRESH = os.environ.get("IMV_FORCE_RECOMPUTE", "0").strip().lower() in {"1", "true", "yes"}
DATA_ROOT, ARTIFACT_ROOT = cache_paths()
print(f"imvpy {imvpy.__version__}; PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}; worker devices: {execution_plan(CONFIG, JOBS, DEVICE)}")
print(f"{len(DATASETS)} datasets x {len(SEEDS)} seeds x {CONFIG.epochs} epochs")
print("Data cache:", DATA_ROOT)
print("Result cache:", ARTIFACT_ROOT)


imvpy 1.2.0; PyTorch 2.9.0+cpu
CUDA available: False; worker devices: ['cpu', 'cpu', 'cpu', 'cpu', 'cpu', 'cpu']
6 datasets x 10 seeds x 200 epochs
Data cache: /home/jinx/.cache/imv/datasets/tu
Result cache: /home/jinx/.cache/imv/notebook_artifacts/gnn_training


## Metrics and seed ranges

**Evaluation is performed at every single epoch**, including epoch 0: 201 validation and 201 test evaluations per seed. The ten-epoch checkpoint interval and 25-epoch console logging interval do not subsample evaluation. The evaluation batches cache graph inputs, never predictions. Each epoch uses the current model in evaluation mode (dropout disabled), with no temporal averaging, parameter averaging, or ensembling across epochs.

**IMV is primary.** Each epoch uses `imvpy.imv_from_likelihoods(ll(y, p_null), ll(y, p_gcn))`. The null predicts the **training-set prevalence**, evaluated on the same held-out graphs as the GCN. The paper does not specify the Section 6 null explicitly; this intercept-only choice reflects class imbalance and is fixed before evaluation. Unlike the ablation overview, it is not a constant 0.5 null. Fair-coin IMV and the null-dependent upper bound are also saved for audit.

Accuracy, binary precision, and binary recall use a fixed threshold of 0.5, following Appendix E.2.2. Positive class is the canonical TU/PyG label 1; the original label order is saved rather than guessing semantic class names. Undefined precision from zero positive predictions is reported as zero. Additional diagnostics include binary F1, ROC AUC, balanced accuracy, Brier score, log loss, macro/weighted precision and recall, geometric likelihood, and information deficit.

The pinned `imvpy` sometimes returns its chance-boundary value for mildly below-chance likelihoods; severely below-chance likelihoods yield NaN. We preserve and flag that behavior. **Negative IMV is retained.** Undefined IMVs are not replaced with zero. A plotted point and its range require all ten finite seed values; incomplete points are gaps and the coverage table reports them.

Lines show the **mean across ten seeds**. The shaded band follows the **minimum and maximum seed values at every epoch** along each mean curve, computed separately for each dataset and metric. Individual seed lines are omitted from the overview. These are observed seed ranges, **not confidence intervals**. A separate IMV figure shows every seed without averaging and uses explicitly labelled dataset-specific y-axis ranges. No smoothing, renderer path simplification, monotonicity constraint, or curve matching is used. The plotter calculates these ranges from the saved per-seed metrics; the original CI description in training manifests is historical and does not control plotting. Training code and checkpoint fingerprints are unchanged.


## Execution and restart behavior

Training is a full 60-run experiment. CPU execution uses bounded, separate seed-worker processes with a fixed thread budget. With CUDA-enabled PyTorch and a working driver, `IMV_GNN_DEVICE=auto` uses one process per visible GPU, distributing independent runs across devices. `CUDA_VISIBLE_DEVICES` selects GPUs; `IMV_GNN_JOBS` can limit workers. A single GPU is not oversubscribed by default.

Graph normalization is cached. Sparse matrix multiplication avoids large edge-message tensors; node-budgeted microbatches accumulate gradients into the same 512-graph optimizer batch. Validation/test batches are reused. Deterministic algorithms are required; unsupported GPU kernels fail visibly rather than silently relaxing reproducibility. No mixed precision is enabled. Bitwise equality between CPU and GPU or across library versions is not promised.

Each job checkpoints model, Adam state, RNG state, epoch history, and test-probability history every ten epochs. Resuming restores the last complete checkpoint. Changes to data, code, model configuration, or numerical environment invalidate a cached job; changing only worker count does not. Concurrent invocations are locked. Completed test-probability traces allow every reported test metric to be audited without refitting.

The defaults produce publication results. Smaller configurations, subsets, or fewer seeds are written under a separate `checks/` directory and cannot overwrite or satisfy the publication plotter. `IMV_FORCE_RECOMPUTE=1` bypasses checkpoints. Do not use `run_all.sh --fresh` unless you intend to rerun every project example.


In [2]:
epoch_metrics, result_directory = run_experiments(
    CONFIG, datasets=DATASETS, seeds=SEEDS, jobs=JOBS, device=DEVICE,
    data_root=DATA_ROOT, artifact_root=ARTIFACT_ROOT, fresh=FRESH,
)
print("Completed GNN results:", result_directory)


GNN: 60 runs x 200 epochs; devices=['cpu', 'cpu', 'cpu', 'cpu', 'cpu', 'cpu']; 2 CPU threads/worker; features=native


PROTEINS seed 44 epoch 0/200: IMV=-0.1649, accuracy=0.4012
PROTEINS seed 46 epoch 0/200: IMV=-0.1565, accuracy=0.4072
PROTEINS seed 42 epoch 0/200: IMV=-0.1332, accuracy=0.6048
PROTEINS seed 45 epoch 0/200: IMV=-0.0462, accuracy=0.5928
PROTEINS seed 47 epoch 0/200: IMV=-0.0991, accuracy=0.6108
PROTEINS seed 43 epoch 0/200: IMV=-0.1649, accuracy=0.4012


PROTEINS seed 45 epoch 25/200: IMV=0.1486, accuracy=0.6527
PROTEINS seed 44 epoch 25/200: IMV=0.0640, accuracy=0.6287
PROTEINS seed 43 epoch 25/200: IMV=0.0629, accuracy=0.6228
PROTEINS seed 47 epoch 25/200: IMV=0.1107, accuracy=0.6347
PROTEINS seed 42 epoch 25/200: IMV=0.0635, accuracy=0.6228


PROTEINS seed 46 epoch 25/200: IMV=0.0831, accuracy=0.6347


PROTEINS seed 44 epoch 50/200: IMV=0.0872, accuracy=0.6587


...

DD seed 50 epoch 200/200: IMV=0.3399, accuracy=0.7740


DD seed 51 epoch 175/200: IMV=0.2957, accuracy=0.7514


DD seed 51 epoch 200/200: IMV=0.3002, accuracy=0.7458


Completed results: /home/jinx/.cache/imv/notebook_artifacts/gnn_training/results


Completed GNN results: /home/jinx/.cache/imv/notebook_artifacts/gnn_training/results


In [3]:
final_test = epoch_metrics.query("split == 'test' and epoch == @CONFIG.epochs")
final_summary = final_test.groupby("dataset", sort=False)[
    ["imv", "accuracy", "precision", "recall", "balanced_accuracy", "roc_auc", "brier", "log_loss"]
].agg(["mean", "std", "count"])
final_summary


imv                  accuracy                 precision  \
                  mean       std count      mean       std count      mean   
dataset                                                                      
PROTEINS      0.212052  0.048759    10  0.711377  0.027772    10  0.674938   
NCI1          0.397838  0.051240    10  0.694003  0.015747    10  0.692744   
NCI109        0.374870  0.037716    10  0.684194  0.019456    10  0.673002   
Mutagenicity  0.438739  0.029058    10  0.757604  0.017556    10  0.725695   
AIDS          0.237739  0.004618    10  0.984000  0.005622    10  0.988840   
DD            0.261581  0.062278    10  0.732203  0.026526    10  0.742695   

                                recall  ... balanced_accuracy   roc_auc  \
                   std count      mean  ...             count      mean   
dataset                                 ...                               
PROTEINS      0.039508    10  0.548310  ...                10  0.734824   
NCI1          0.023564    10  0.702589  ...                10  0.747890   
NCI109        0.027966    10  0.730128  ...                10  0.732406   
Mutagenicity  0.025872    10  0.734828  ...                10  0.836639   
AIDS          0.006920    10  0.991250  ...                10  0.993840   
DD            0.043014    10  0.536986  ...                10  0.761301   

                                 brier                  log_loss            \
                   std count      mean       std count      mean       std   
dataset                                                                      
PROTEINS      0.036154    10  0.200066  0.011456    10  0.587760  0.026792   
NCI1          0.012689    10  0.204864  0.004136    10  0.609488  0.020595   
NCI109        0.015845    10  0.210973  0.005600    10  0.616893  0.015546   
Mutagenicity  0.015873    10  0.164066  0.007618    10  0.502693  0.022214   
AIDS          0.005912    10  0.012072  0.004109    10  0.054440  0.017378   
DD            0.040599    10  0.190149  0.013925    10  0.568681  0.035235   

                    
             count  
dataset             
PROTEINS        10  
NCI1            10  
NCI109          10  
Mutagenicity    10  
AIDS            10  
DD              10  

[6 rows x 24 columns]

In [4]:
imv_coverage = epoch_metrics.groupby(["dataset", "split"])["imv_status"].value_counts().unstack(fill_value=0)
imv_coverage


imv_status               chance_boundary  defined
dataset      split                               
AIDS         test                      8     2002
             validation                8     2002
DD           test                     12     1998
             validation               12     1998
Mutagenicity test                      8     2002
             validation                8     2002
NCI1         test                     11     1999
             validation               12     1998
NCI109       test                     13     1997
             validation               13     1997
PROTEINS     test                     15     1995
             validation               14     1996

## Rebuild only the GNN experiment and publication plots

From the repository root, in an environment with `requirements.txt` installed:

```bash
IMV_FORCE_RECOMPUTE=0 jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=-1 src/empirical/gnn/gnn_training.ipynb
jupyter nbconvert --execute --to notebook --inplace src/plotter/plotter.ipynb
```

The second command only reads saved results. It produces PDF figures with Helvetica and `Spectral_r`. No PNG/SVG figure files are saved. The existing ablation CSVs are also required for the combined plotter.
